<a href="https://colab.research.google.com/github/saroshk/Test/blob/master/Lab_18_Reusable_Skills_Prompt_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 18: Reusable Skills & Prompt Optimization

Build **reusable skill packs** — specialized instruction sets that an agent loads on-demand based on query classification. Then **A/B test** the skill-based agent against a generic one, and **iteratively optimize** the weakest skill using LLM-as-Judge feedback.

## Business Scenario: TechNova Solutions — Enterprise AI Assistant

**TechNova Solutions** is a mid-size Indian enterprise handling three distinct workflows: insurance claims processing, customer support, and regulatory compliance. Their single AI assistant struggles:

1. **Quality Varies Wildly (2.9-4.2/5):** One generic prompt tries to handle 3 very different domains. Claims processing scores 4.2/5, support 3.8/5, but compliance review only 2.9/5 — barely usable.

2. **Prompt Maintenance Nightmare (2 hrs/week):** Domain experts spend 2 hours weekly updating prompts scattered across 15 config files. Changes break other domains because everything shares one giant prompt.

3. **No Testing Discipline (\"Prompt YOLO\"):** Prompt changes go live without testing. Last month, a well-intentioned tweak to the claims prompt caused 300 compliance reviews to get wrong instructions.

4. **Token Waste (\u20b94,200/day):** Loading all 3 domain instructions (2000 tokens) every turn when only 1 domain is needed (600 tokens). At 500 queries/day, that's \u20b94,200/day in wasted tokens.

**Our Solution:** Modular skills (specialized instruction packs) loaded on-demand. A/B test each skill systematically. Iteratively optimize based on measured results.

In [ ]:
%%capture
!pip install langchain langchain-openai langgraph

In [ ]:
# --- API Keys (Google Colab Secrets) ---
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# --- Imports ---
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
import json
import copy

# --- LLM ---
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

print("Imports loaded. Model: gpt-4.1-mini (temp=0.3)")

Imports loaded. Model: gpt-4.1-mini (temp=0.3)


## What Are Skills? Reusable Instruction Packs

A **skill** is a self-contained instruction set for a specific domain. Instead of one giant system prompt that covers everything poorly, the agent loads **only the relevant skill** for the current task. Think of it as progressive disclosure — load information on-demand, not upfront.

In [ ]:
# --- Define 3 Skills as Python Dictionaries ---
# Each skill has: name, description, system_prompt, rules, few_shot_examples

SKILLS_REGISTRY = {
    "claim_processing": {
        "name": "Insurance Claims Processing",
        "description": "Process and evaluate insurance claims following IRDAI guidelines",
        "system_prompt": (
            "You are TechNova's Claims Processing Specialist. You evaluate insurance claims "
            "following IRDAI (Insurance Regulatory and Development Authority of India) guidelines.\n\n"
            "When processing a claim:\n"
            "1. Verify all required documents are mentioned (policy number, incident date, claim amount)\n"
            "2. Check for fraud indicators (inconsistent dates, inflated amounts, repeated claims)\n"
            "3. Apply threshold rules: Auto-approve below \u20b950,000; escalate above \u20b95,00,000\n"
            "4. Provide a clear recommendation: APPROVE / DENY / ESCALATE with reasoning"
        ),
        "rules": (
            "- ALWAYS mention the policy number in your response\n"
            "- Claims older than 90 days require manager review\n"
            "- Water damage claims need photos as supporting evidence\n"
            "- Never approve claims without incident date verification"
        ),
        "few_shot_examples": [
            {
                "query": "Claim for \u20b945,000 by Rajesh Kumar, policy INS-2024-789, water damage to electronics on 15 March 2026.",
                "response": "Claim Review for Policy INS-2024-789:\n- Amount: \u20b945,000 (within auto-approval threshold)\n- Type: Water damage to electronics\n- Date: 15 March 2026 (within 90-day window)\n- Required: Please confirm supporting photos of water damage are on file.\nRecommendation: CONDITIONAL APPROVE pending photo verification."
            }
        ]
    },

    "customer_support": {
        "name": "Customer Support",
        "description": "Handle customer queries, complaints, and service requests",
        "system_prompt": (
            "You are TechNova's Customer Support Specialist. You handle customer queries "
            "with empathy, efficiency, and adherence to SLA commitments.\n\n"
            "SLA Commitments:\n"
            "- P1 (Critical): Respond within 4 hours, resolve within 24 hours\n"
            "- P2 (Standard): Respond within 24 hours, resolve within 72 hours\n"
            "- P3 (Low): Respond within 48 hours, resolve within 1 week\n\n"
            "Escalation triggers: Customer waiting >48 hours, repeated contacts (3+), "
            "explicit request for manager, mentions of legal action."
        ),
        "rules": (
            "- ALWAYS acknowledge the customer's frustration before offering solutions\n"
            "- Provide specific SLA timelines, don't say 'soon'\n"
            "- Offer compensation for SLA breaches (\u20b9500 credit or priority handling)\n"
            "- Never blame the customer; use 'we' language"
        ),
        "few_shot_examples": [
            {
                "query": "I've been waiting 3 days for a response about my policy renewal. This is unacceptable.",
                "response": "I sincerely apologize for the delay, and I understand your frustration. A 3-day wait exceeds our P2 SLA of 24-hour response time. Here's what I'm doing right now:\n1. Escalating your renewal to our priority queue\n2. Applying a \u20b9500 service credit to your account\n3. You will receive an update within 4 hours\nIs there anything else I can help with?"
            }
        ]
    },

    "compliance_review": {
        "name": "Regulatory Compliance Review",
        "description": "Review processes and documents for IRDAI regulatory compliance",
        "system_prompt": (
            "You are TechNova's Compliance Review Specialist. You audit processes and documents "
            "for compliance with IRDAI regulations and internal policies.\n\n"
            "Compliance Framework:\n"
            "1. Document Completeness: All required forms present and filled\n"
            "2. Data Accuracy: Policyholder data matches records\n"
            "3. Timeline Compliance: Processing within regulatory deadlines (30 days for claims)\n"
            "4. Audit Trail: Every decision documented with reason and approver"
        ),
        "rules": (
            "- ALWAYS reference specific IRDAI regulation numbers when citing rules\n"
            "- Flag any missing documentation explicitly\n"
            "- Use severity levels: CRITICAL / HIGH / MEDIUM / LOW for findings\n"
            "- Include recommended remediation for each finding"
        ),
        "few_shot_examples": [
            {
                "query": "Review compliance status of claim batch BT-2026-Q1.",
                "response": "Compliance Review: Batch BT-2026-Q1\n\nFindings:\n1. [CRITICAL] 3 of 50 claims missing incident date documentation (IRDAI Regulation 17.2)\n2. [HIGH] 8 claims processed beyond 30-day deadline (IRDAI Regulation 14.1)\n3. [MEDIUM] 12 claims lack digital signature on approval form\n\nRemediation:\n- Immediately return 3 incomplete claims for documentation\n- File delay justification for 8 overdue claims\n- Implement digital signature workflow by next quarter"
            }
        ]
    }
}

print(f"Skills registry: {len(SKILLS_REGISTRY)} skills")
for name, skill in SKILLS_REGISTRY.items():
    tokens = len(skill['system_prompt'].split()) * 1.3  # Rough token estimate
    print(f"  {name}: ~{int(tokens)} tokens")

In [ ]:
# --- Token comparison: All skills vs. single skill ---
all_skills_text = "\n\n".join([s["system_prompt"] + "\n" + s["rules"] for s in SKILLS_REGISTRY.values()])
single_skill_text = SKILLS_REGISTRY["claim_processing"]["system_prompt"] + "\n" + SKILLS_REGISTRY["claim_processing"]["rules"]

print(f"All 3 skills combined: ~{len(all_skills_text.split())} words")
print(f"Single skill loaded: ~{len(single_skill_text.split())} words")
print(f"Savings per query: ~{(1 - len(single_skill_text.split())/len(all_skills_text.split()))*100:.0f}% fewer tokens")

## Approach 1: Agent Loads Skills via Tool Call

The agent has a `load_skill` tool. When a query arrives, the agent **decides** which skill to load and calls the tool. The skill instructions flow into the conversation as a tool result.

In [ ]:
# --- Approach 1: Tool-Based Skill Loading ---
# The agent calls load_skill() as its first action, then follows the loaded instructions

def load_skill(skill_name: str) -> str:
    """Load a specialized skill for the current task."""
    skill = SKILLS_REGISTRY.get(skill_name)
    if not skill:
        return f"Unknown skill: {skill_name}. Available: {list(SKILLS_REGISTRY.keys())}"
    examples_text = "\n".join([
        f"Example Query: {ex['query']}\nExample Response: {ex['response']}"
        for ex in skill["few_shot_examples"]
    ])
    return (
        f"SKILL LOADED: {skill['name']}\n\n"
        f"INSTRUCTIONS:\n{skill['system_prompt']}\n\n"
        f"RULES:\n{skill['rules']}\n\n"
        f"EXAMPLES:\n{examples_text}"
    )


# Simulate tool-based approach using a two-step chain
skill_agent_prompt = ChatPromptTemplate.from_template(
    """You are TechNova's enterprise assistant. For EVERY task, first determine which skill is needed,
then follow the loaded skill instructions exactly.

Loaded Skill:
{skill_context}

Customer Query: {query}

Respond following the loaded skill's instructions and rules precisely.""")

skill_chain = skill_agent_prompt | llm

def run_skill_agent_v1(query: str) -> str:
    """Run the tool-based skill agent. Classify query → load skill → respond."""
    # Step 1: Classify which skill is needed
    q = query.lower()
    if any(kw in q for kw in ["claim", "damage", "policy holder", "incident", "insurance"]):
        skill_name = "claim_processing"
    elif any(kw in q for kw in ["waiting", "complaint", "support", "help", "frustrated", "issue"]):
        skill_name = "customer_support"
    elif any(kw in q for kw in ["compliance", "audit", "irdai", "regulation", "review batch"]):
        skill_name = "compliance_review"
    else:
        skill_name = "customer_support"  # Default

    # Step 2: Load the skill
    skill_context = load_skill(skill_name)

    # Step 3: Generate response with skill context
    response = skill_chain.invoke({"skill_context": skill_context, "query": query}).content
    return response, skill_name

print("Approach 1 ready: Tool-based skill loading")

In [ ]:
# --- Test Approach 1 with 3 Queries ---
test_queries_v1 = [
    "Process this claim: Policy holder Rajesh Kumar, policy INS-2024-789, claim amount \u20b945,000 for water damage to electronics on 15 March 2026.",
    "I have been waiting 3 days for a response about my policy renewal. This is unacceptable. Customer ID CUST-456.",
    "Review the compliance status of claim batch BT-2026-Q1. Check if all IRDAI documentation requirements are met.",
]

print("Approach 1: Tool-Based Skill Agent")
print("=" * 60)
for i, query in enumerate(test_queries_v1, 1):
    response, skill_used = run_skill_agent_v1(query)
    print(f"\nQuery {i}: {query[:60]}...")
    print(f"Skill loaded: {skill_used}")
    print(f"Response: {response[:300]}...")
    print("-" * 40)

## Approach 2: Dynamic System Prompt (More Efficient)

Instead of loading skills via a tool call (extra LLM round-trip), we **swap the system prompt itself** before the model call. The classifier runs first, then the correct skill is injected directly into the prompt. This is faster and enables prompt caching (the prefix stays stable).

In [ ]:
# --- Approach 2: Dynamic System Prompt ---

def classify_task(query: str) -> str:
    """Classify which skill is needed (keyword-based, no LLM call)."""
    q = query.lower()
    if any(kw in q for kw in ["claim", "damage", "policy holder", "incident", "insurance", "policy number"]):
        return "claim_processing"
    elif any(kw in q for kw in ["waiting", "complaint", "support", "help", "frustrated", "renewal"]):
        return "customer_support"
    elif any(kw in q for kw in ["compliance", "audit", "irdai", "regulation", "review batch", "review the"]):
        return "compliance_review"
    return "customer_support"  # Default


def build_skill_prompt(query: str) -> str:
    """Build a system prompt with the relevant skill loaded."""
    skill_name = classify_task(query)
    skill = SKILLS_REGISTRY[skill_name]

    # Stable prefix (same across all skills — enables prompt caching)
    base = "You are TechNova's enterprise AI assistant.\n\n"
    base += f"ACTIVE SKILL: {skill['name']}\n\n"
    base += f"INSTRUCTIONS:\n{skill['system_prompt']}\n\n"
    base += f"RULES:\n{skill['rules']}\n\n"

    # Add few-shot examples
    for ex in skill["few_shot_examples"]:
        base += f"EXAMPLE:\nQuery: {ex['query']}\nResponse: {ex['response']}\n\n"

    return base, skill_name


# Dynamic prompt chain
dynamic_prompt = ChatPromptTemplate.from_template(
    "{system_prompt}\n\nCustomer Query: {query}\n\nRespond following the loaded skill's instructions and rules precisely.")
dynamic_chain = dynamic_prompt | llm


def run_skill_agent_v2(query: str) -> str:
    """Run the dynamic-prompt skill agent."""
    system_prompt, skill_name = build_skill_prompt(query)
    response = dynamic_chain.invoke({"system_prompt": system_prompt, "query": query}).content
    return response, skill_name


print("Approach 2 ready: Dynamic system prompt")
print("Advantage: No extra tool call. System prompt swaps per query.")

In [ ]:
# --- Test Approach 2 with Same 3 Queries ---
print("Approach 2: Dynamic System Prompt Agent")
print("=" * 60)
for i, query in enumerate(test_queries_v1, 1):
    response, skill_used = run_skill_agent_v2(query)
    print(f"\nQuery {i}: {query[:60]}...")
    print(f"Skill loaded: {skill_used}")
    print(f"Response: {response[:300]}...")
    print("-" * 40)

## A/B Test: Skill Agent vs Generic Agent

To prove skills actually help, we run the **same 10 queries** through two agents: one with skills and one with a generic prompt. An **LLM-as-Judge** scores each response on accuracy (1-5), completeness (1-5), and tone (1-5).

In [ ]:
# --- LLM-as-Judge Scorer ---
judge_prompt = ChatPromptTemplate.from_template(
    """You are an expert evaluator for enterprise AI responses.

Score this response on three dimensions (1-5 each):
1. **Accuracy**: Does it follow domain-specific rules correctly?
2. **Completeness**: Does it address all aspects of the query?
3. **Tone**: Is the tone professional and appropriate for the domain?

Query: {query}
Response: {response}
Domain: {domain}

Return ONLY a JSON object: {{"accuracy": X, "completeness": Y, "tone": Z}}""")

judge_chain = judge_prompt | llm


def score_response(query: str, response: str, domain: str) -> dict:
    """Score a response using LLM-as-Judge."""
    try:
        result = judge_chain.invoke({"query": query, "response": response, "domain": domain}).content
        # Parse JSON from response
        result = result.strip()
        if result.startswith("```"):
            result = result.split("```")[1].replace("json", "").strip()
        scores = json.loads(result)
        scores["total"] = scores["accuracy"] + scores["completeness"] + scores["tone"]
        return scores
    except:
        return {"accuracy": 3, "completeness": 3, "tone": 3, "total": 9}

print("LLM-as-Judge scorer ready (accuracy + completeness + tone, each 1-5)")

In [ ]:
# --- 10 Test Queries (Mixed Domains, Indian Context) ---
TEST_QUERIES = [
    # Claims (4 queries)
    {"query": "Process this claim: Policyholder Sunita Reddy, policy INS-2024-456, claim \u20b938,000 for water damage to laptop on 20 March 2026.", "domain": "claims"},
    {"query": "Claim submission: Amit Joshi, policy INS-2023-112, \u20b97,50,000 for fire damage to office equipment on 1 Feb 2026. All documents attached.", "domain": "claims"},
    {"query": "Check claim status for policy INS-2024-789. The incident happened 95 days ago.", "domain": "claims"},
    {"query": "New claim: Deepa Nair, policy INS-2025-001, \u20b91,20,000 for theft of jewelry. No police report filed yet.", "domain": "claims"},
    # Support (3 queries)
    {"query": "I've been trying to reach someone about my policy renewal for 5 days. No response. This is terrible service.", "domain": "support"},
    {"query": "My claim was approved 2 weeks ago but I haven't received the payout. Please help. Customer ID CUST-789.", "domain": "support"},
    {"query": "I want to speak to a manager. Your automated system keeps disconnecting my calls.", "domain": "support"},
    # Compliance (3 queries)
    {"query": "Review compliance status of claims batch BT-2026-Q1. We need this for the IRDAI quarterly audit.", "domain": "compliance"},
    {"query": "Check if our claims processing timeline meets IRDAI Regulation 14.1 requirements for the last 100 claims.", "domain": "compliance"},
    {"query": "Audit trail review: Verify that all claim approvals above \u20b91,00,000 have dual authorization as per internal policy CP-007.", "domain": "compliance"},
]

print(f"Test queries defined: {len(TEST_QUERIES)} queries")
print(f"  Claims: {sum(1 for q in TEST_QUERIES if q['domain'] == 'claims')}")
print(f"  Support: {sum(1 for q in TEST_QUERIES if q['domain'] == 'support')}")
print(f"  Compliance: {sum(1 for q in TEST_QUERIES if q['domain'] == 'compliance')}")

In [ ]:
# --- Generic Agent (No Skills — Baseline) ---
generic_prompt = ChatPromptTemplate.from_template(
    """You are TechNova's enterprise AI assistant. Help with insurance claims, customer support, and compliance reviews.

Customer Query: {query}

Provide a helpful, professional response.""")

generic_chain = generic_prompt | llm

def run_generic_agent(query: str) -> str:
    """Run the generic agent (no skills)."""
    return generic_chain.invoke({"query": query}).content

print("Generic agent ready (single prompt, no skills)")

In [ ]:
# --- Run A/B Test ---
print("Running A/B Test: Generic vs Skill Agent")
print("=" * 70)

results = []
for i, test in enumerate(TEST_QUERIES, 1):
    query = test["query"]
    domain = test["domain"]

    # Generic agent
    generic_response = run_generic_agent(query)
    generic_scores = score_response(query, generic_response, domain)

    # Skill agent (Approach 2 — dynamic prompt)
    skill_response, skill_used = run_skill_agent_v2(query)
    skill_scores = score_response(query, skill_response, domain)

    results.append({
        "query_num": i,
        "domain": domain,
        "skill_used": skill_used,
        "generic_total": generic_scores["total"],
        "skill_total": skill_scores["total"],
        "delta": skill_scores["total"] - generic_scores["total"],
        "generic_scores": generic_scores,
        "skill_scores": skill_scores,
    })

    print(f"Q{i} [{domain:>10}]: Generic={generic_scores['total']:>2}/15 | Skill={skill_scores['total']:>2}/15 | Delta={skill_scores['total'] - generic_scores['total']:>+3}")

# Summary
avg_generic = sum(r["generic_total"] for r in results) / len(results)
avg_skill = sum(r["skill_total"] for r in results) / len(results)
print(f"\n{'='*70}")
print(f"Average Generic: {avg_generic:.1f}/15 | Average Skill: {avg_skill:.1f}/15 | Improvement: {avg_skill - avg_generic:+.1f}")

## Optimize the Weakest Skill

The A/B test reveals which domain performs worst. We now analyze failures and iteratively improve that skill.

In [ ]:
# --- Identify Worst-Performing Skill ---
domain_scores = {}
for r in results:
    domain = r["domain"]
    if domain not in domain_scores:
        domain_scores[domain] = []
    domain_scores[domain].append(r["skill_total"])

print("Skill Performance by Domain:")
worst_domain = None
worst_avg = 999
for domain, scores in domain_scores.items():
    avg = sum(scores) / len(scores)
    print(f"  {domain}: avg {avg:.1f}/15 ({len(scores)} queries)")
    if avg < worst_avg:
        worst_avg = avg
        worst_domain = domain

# Map domain to skill name
domain_to_skill = {"claims": "claim_processing", "support": "customer_support", "compliance": "compliance_review"}
worst_skill_name = domain_to_skill[worst_domain]
print(f"\nWeakest skill: {worst_skill_name} (avg {worst_avg:.1f}/15)")

In [ ]:
# --- Analyze Failures and Optimize ---
optimization_prompt = ChatPromptTemplate.from_template(
    """Analyze these AI responses that scored poorly and suggest improvements to the skill prompt.

Skill Name: {skill_name}
Current System Prompt: {current_prompt}
Current Rules: {current_rules}

Poor-performing responses:
{failures}

Suggest exactly:
1. 2-3 additional rules to add (be specific)
2. 1 improved few-shot example
3. Any tone/style adjustments

Return your suggestions as a clear numbered list.""")

optimization_chain = optimization_prompt | llm

# Get the failures for the worst skill
worst_results = [r for r in results if r["domain"] == worst_domain]
failures_text = "\n".join([
    f"Query: {TEST_QUERIES[r['query_num']-1]['query'][:100]}\n"
    f"Scores: accuracy={r['skill_scores']['accuracy']}, completeness={r['skill_scores']['completeness']}, tone={r['skill_scores']['tone']}"
    for r in worst_results
])

worst_skill = SKILLS_REGISTRY[worst_skill_name]
suggestions = optimization_chain.invoke({
    "skill_name": worst_skill_name,
    "current_prompt": worst_skill["system_prompt"],
    "current_rules": worst_skill["rules"],
    "failures": failures_text,
}).content

print(f"Optimization suggestions for '{worst_skill_name}':")
print(suggestions)

In [ ]:
# --- Apply Improvements: Create V2 of Worst Skill ---
SKILLS_REGISTRY_V2 = copy.deepcopy(SKILLS_REGISTRY)

# Add optimization suggestions to the worst skill's rules
SKILLS_REGISTRY_V2[worst_skill_name]["rules"] += (
    "\n- ALWAYS provide specific next steps with timelines\n"
    "- Reference specific policy/regulation numbers in every response\n"
    "- Include a severity assessment (CRITICAL/HIGH/MEDIUM/LOW) for compliance items"
)

# Store V2 for re-testing
# Swap the registry temporarily
original_registry = SKILLS_REGISTRY

print(f"V2 of '{worst_skill_name}' created with additional rules.")
print(f"\nV1 rules ({len(worst_skill['rules'].split(chr(10)))} lines):")
print(worst_skill['rules'][:200])
print(f"\nV2 rules ({len(SKILLS_REGISTRY_V2[worst_skill_name]['rules'].split(chr(10)))} lines):")
print(SKILLS_REGISTRY_V2[worst_skill_name]['rules'][:300])

In [ ]:
# --- Re-run A/B with Optimized Skill ---
# Temporarily swap to V2 registry
SKILLS_REGISTRY_BACKUP = SKILLS_REGISTRY
globals()['SKILLS_REGISTRY'] = SKILLS_REGISTRY_V2

print(f"Re-running queries for '{worst_domain}' domain with optimized skill...")
print("=" * 60)

v2_scores = []
for r in worst_results:
    query = TEST_QUERIES[r["query_num"]-1]["query"]
    response_v2, _ = run_skill_agent_v2(query)
    scores_v2 = score_response(query, response_v2, worst_domain)
    v2_scores.append(scores_v2["total"])
    print(f"Q{r['query_num']}: V1={r['skill_total']}/15 -> V2={scores_v2['total']}/15 ({scores_v2['total'] - r['skill_total']:+d})")

# Restore original registry
globals()['SKILLS_REGISTRY'] = SKILLS_REGISTRY_BACKUP

v1_avg = sum(r["skill_total"] for r in worst_results) / len(worst_results)
v2_avg = sum(v2_scores) / len(v2_scores)
print(f"\nImprovement: {v1_avg:.1f}/15 -> {v2_avg:.1f}/15 ({v2_avg - v1_avg:+.1f} points)")

## Bonus: How OpenAI Agents SDK Handles Skills

In Labs 13/14, we used the OpenAI SDK. The skills pattern maps differently in that framework:

In [ ]:
# --- OpenAI SDK Skills Equivalent (REFERENCE CODE) ---

OPENAI_SDK_SKILLS = '''
# In OpenAI Agents SDK, "skills" map to SEPARATE AGENTS with HANDOFFS.
# Instead of one agent loading skills, you define specialist agents.

# LangGraph approach (this lab):
# One agent + dynamic prompt that swaps skill per query

# OpenAI SDK approach (Labs 13/14):
# Triage agent -> handoff to specialist agent (each has its own instructions)

from agents import Agent, Runner

# Each "skill" becomes a dedicated agent
claims_agent = Agent(
    name="Claims Specialist",
    instructions="You are TechNova's Claims Processing Specialist...",  # = skill prompt
    tools=[check_claim_docs, calculate_fraud_risk],
)

support_agent = Agent(
    name="Support Specialist",
    instructions="You are TechNova's Customer Support Specialist...",
    tools=[lookup_ticket, check_sla],
)

# Triage agent routes to specialists (= skill selection)
triage = Agent(
    name="TechNova Assistant",
    instructions="Route to the appropriate specialist...",
    handoffs=[claims_agent, support_agent, compliance_agent],
)

# For prompt optimization, the OpenAI Cookbook uses:
# Multi-agent validation pipeline (contradiction detection, format checking)
# See: developers.openai.com/cookbook/examples/optimize_prompts
'''

print("OpenAI SDK Skills Pattern Comparison:")
print("  LangGraph: One agent + dynamic prompt that swaps per query")
print("  OpenAI SDK: Multiple agents + triage handoffs")
print("  Both achieve the same result: specialized behavior per domain")
print("\n  LangGraph advantage: One agent, less overhead, dynamic switching")
print("  OpenAI SDK advantage: Each specialist is fully independent, can have its own tools")

### Skills Pattern: LangGraph vs OpenAI SDK

| Aspect | LangGraph (this lab) | OpenAI Agents SDK (Labs 13/14) |
|--------|---------------------|-------------------------------|
| **Skill = ?** | Python dict with prompt + rules | Separate `Agent(instructions=)` |
| **Skill loading** | Dynamic prompt swap or tool call | Handoff from triage agent |
| **Skill selection** | Keyword classifier or LLM | LLM decides handoff target |
| **Quality testing** | LLM-as-Judge A/B test | Multi-agent validation (cookbook) |
| **Caching benefit** | Stable prefix enables KV reuse | `prompt_cache_key` parameter |
| **When to use** | Few skills, fast switching needed | Many specialists, each needs own tools |

## Conclusion

In this lab, we built a **reusable skills system** for TechNova Solutions:
- **3 skills** (Claims Processing, Customer Support, Compliance Review) as Python dicts
- **Approach 1:** Tool-based skill loading (agent calls `load_skill`)
- **Approach 2:** Dynamic prompt (system prompt swaps per query — more efficient)
- **A/B test:** Skill agent scored higher than generic agent across 10 queries
- **Iterative optimization:** Identified and improved the weakest skill using LLM-as-Judge

### Key Takeaways
- Skills = modular prompts loaded on-demand (progressive disclosure)
- Always A/B test before deploying prompt changes (no more "prompt YOLO")
- Dynamic prompts enable prompt caching (stable prefix, variable skill section)

### Next Lab Preview
In **Lab 19**, we build a comprehensive **evaluation suite** for the ShopSmart Spine A agent — combining unit tests, LLM-as-Judge, and trajectory evaluation into a 3-layer eval hierarchy.